In [1]:
#TODO make sure this renders in the github repo

# Llama 3 Tokenizer

🌟 A tokenizer takes raw text strings and outputs a 1D list of discrete integers/tokens (e.g., "The brown rabbit..." becomes [1, 45, 102, ...]).


**Useful Links:**

- [Tokenization Visualization](https://tiktokenizer.vercel.app/?model=meta-llama%2FMeta-Llama-3-8B) 
- [HuggingFace Byte-Pair Encoding tokenization explanation](https://huggingface.co/learn/llm-course/chapter6/5)

**Llama 3 paper:** "We use a vocabulary with $\bold{128K}$ tokens. Our token vocabulary combines $\bold{100K}$ tokens from the **tiktoken tokenizer** with $\bold{28K}$ additional tokens to better support non-English languages. Compared to the Llama $2$ tokenizer, our new tokenizer improves compression rates on a sample of English data from $3.17$ to $3.94$ characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding $\bold{28K}$ tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

- They took OpenAI's pre-trained tokenizer, then they got their own dataset consisting of non-English languages and code, ran a BPE training algorithm on that dataset to generate an additional 28K BPE merge rules, and finally combined the pre-trained tokenizer with the 28K tokens to get the Llama 3 tokenizer.
- Unfortunately for my scaled down model I can not use OpenAI's pre-trained tokenizer, so I will need to train a new BPE tokenizer.

**Goals:**

- [x] Use HuggingFace's Tokenizer library to create a tokenizer that will be trained on a the FineWeb-edu dataset.

In [2]:
import easyjupyter

In [3]:
# @i-c
%load_ext autoreload
%autoreload 2

In [4]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing
import json

In [5]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:    # from llama_configs import BaseConfig
    from llama_configs import BaseConfig

In [6]:
class BPETokenizer:
    def __init__(self, cfg: BaseConfig):
        """Handles training, saving and loading a custom BPE tokenizer for my scaled down Llama model. Not to be used with the trained Llama 3 models, they have their own tokenizers."""
        self.cfg = cfg
        self.tokenizer_dir = cfg.ARTIFACTS_DIR/ "universal_tokenizer"

    def _get_path(self, samples: int):
        """Returns the paths for the tokenizer file and info file."""
        base_name = f"bpe_vocab_size_{self.cfg.vocab_size}_samples_{samples}"
        return (
            self.tokenizer_dir / f"{base_name}.json",
            self.tokenizer_dir / f"{base_name}_info.json",
        )

    def get_sample_size(self, max_docs: int):
        """The tokenizer does not need to be trained on the same of of documents as the model, 5% of the total dataset is enough. However, you don't want it to scale. For example, 5% of 1B documents if way more than you need to train a tokenizer."""
        if max_docs <= 5000:
            return max_docs
        elif max_docs <= 1_000_000:
            return int(max_docs * 0.05)
        elif max_docs <= 100_000_000:
            return int(max_docs * 0.01)
                

    def train_tokenizer(self, local_dataset) -> Tokenizer:
        """
        Train a custom BPE tokenizer.
        """
        chunk_size = 1000  # The number of strings that is sent to the Rust backend per iteration.
        special_tokens = self.cfg.special_tokens

        training_samples = self.get_sample_size(self.cfg.max_docs)

        def batch_iterator():
            batch = []

            # Slice what we need to train on
            for i in range(min(len(local_dataset), training_samples)):
                batch.append(local_dataset[i]["text"])
                if len(batch) == chunk_size:
                    yield batch
                    batch = []
            if batch:
                # Yield any remaining items
                yield batch

        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)
        tokenizer_path, info_path = self._get_path(training_samples)

        # Initialize a new Byte-Pair-Encoding tokenizer model.
        tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]["token"]))
        # Using Byte-Level pre-tokenization ensures that spaces and punctuation are handled correctly without losing characters.
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        # Configure the BPE trainer
        trainer = BpeTrainer(
            vocab_size=self.cfg.vocab_size,
            special_tokens=[v["token"] for v in special_tokens.values()],
            initial_alphabet=ByteLevel.alphabet(),
        )

        # === Train the tokenizer on the raw text file
        print(
            f"Training the BPE tokenizer | Max Document Count: {training_samples} | chunk_size: {chunk_size}...\n"
        )
        tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

        # === Set up the post-processor to automatically add the begin_of_text token to sequences
        tokenizer.post_processor = TemplateProcessing(
            single=f"{special_tokens['doc_begin_token']["token"]} $A",
            pair=f"{special_tokens['doc_begin_token']["token"]} $A $B",
            special_tokens=[
                (
                    special_tokens["doc_begin_token"]["token"],
                    tokenizer.token_to_id(special_tokens["doc_begin_token"]["token"]),
                )
            ],
        )
        # Tell the tokenizer how to decode the token ids back into text, example: 'Ġ' back to ' '
        tokenizer.decoder = ByteLevelDecoder()

        # === Save the trained tokenizer to a JSON file
        tokenizer.save(str(tokenizer_path))
        print(f"Tokenizer saved to {tokenizer_path}")

        with open(info_path, "w") as f:
            config_dict = {
                "name": tokenizer_path.name,
                "description": "Custom universal tokenizer that is be to be used for all my scaled down models, do not use with any of the Llama pre-trained models like Llama 3 8B",
                "vocab_size": self.cfg.vocab_size,
                "training_samples": training_samples,
                "chunk_size": chunk_size,
            }
            json.dump(config_dict, f, indent=4)

        return tokenizer

    def load_tokenizer(self) -> tuple[bool, Tokenizer]:
        """Load a tokenizer based on the vocab size and the maximum number of documents (training_samples) it was trained on."""
        training_samples = self.get_sample_size(self.cfg.max_docs)
        path, _ = self._get_path(training_samples)

        if path.exists():
            print(
                f"\nLoading existing trained BPE Tokenizer with vocab size: {self.cfg.vocab_size} "
                f"(Trained on {training_samples} documents) from: ({path})..."
            )
            tokenizer = Tokenizer.from_file(str(path))
            return True, tokenizer
        return False, None

### Test: BPETokenizer()

In [7]:
# @i-c
from llama_configs import Llama3_scaled_down
from datasets import load_dataset

cfg = Llama3_scaled_down()
cfg.max_docs = cfg.overfit_max_docs
cfg.vocab_size = 32_000
cfg.token_budget = 250 * cfg.global_batch_size_tokens

raw_parquet_file = cfg.DATA_DIR / "overfit_initial_and_lc_stage.parquet"

if not raw_parquet_file.exists():
    raise FileNotFoundError(
        f"The dataset file {raw_parquet_file} does not exist. \n\nPlease run `python prepare.py --d pretrain --overfit` first."
    )

local_ds = load_dataset(
    "parquet", data_files=str(raw_parquet_file), split="train"
)

t = BPETokenizer(cfg)
print("\n\nTesting training a tokenizer...")
trained_tokenizer = t.train_tokenizer(local_ds)

/Users/tonyavis/miniconda3/envs/build_an_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Project Root: /Users/tonyavis/Main/Build_an_LLM


Testing training a tokenizer...
Training the BPE tokenizer | Max Document Count: 1000 | chunk_size: 1000...




Tokenizer saved to /Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_32000_samples_1000.json


In [8]:
# @i-c
# Test: Loading A Trained Tokenizer
success, loaded_tokenizer = t.load_tokenizer()
if success:
    print("Tokenizer successfully trained!")

    print("\n\nEncoding a string to IDs")
    text_str = "The brown rabbit ate the apple."
    encoded = loaded_tokenizer.encode(text_str)
    print(f"Original string: {text_str}")
    print(f"Token IDs: {encoded.ids}")
    print(f"Tokens: {encoded.tokens}")

    print("\n\nDecoding IDs to a string")
    decoded = loaded_tokenizer.decode(encoded.ids)
    print(f"Decoded string: {decoded}")


Loading existing trained BPE Tokenizer with vocab size: 32000 (Trained on 1000 documents) from: (/Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_32000_samples_1000.json)...
Tokenizer successfully trained!


Encoding a string to IDs
Original string: The brown rabbit ate the apple.
Token IDs: [1, 449, 5690, 7011, 1585, 12317, 265, 9782, 17]
Tokens: ['<|begin_of_doc|>', 'The', 'Ġbrown', 'Ġrab', 'bit', 'Ġate', 'Ġthe', 'Ġapple', '.']


Decoding IDs to a string
Decoded string: The brown rabbit ate the apple.
